---
# IMPORTS

In [1]:
import sys, os
sys.path.insert(0, os.path.join('..'))   # project root on path

import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import wrds
import polars as pl
import polars.selectors as cs
import pyarrow
import config
import password

from src.data_loading import *
from src.preprocessing import clean_crsp, clean_futures
from src.feature_engineering import add_target, add_volatility_momentum, crosssectional_rank, get_feature_cols, polars_features
from src.utils import *

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.float_format', '{:.4f}'.format)

---
# Load daily dataset and Feature engineering

In [ ]:

data = pd.read_parquet(config.CRSP_PATH_CLEAN)
data = data.sort_values(['PERMNO', 'date']).reset_index(drop=True)
data['date'] = pd.to_datetime(data['date'])
print(data.shape)
print(data.columns.tolist())
print(data.dtypes)
print(data.head())


---
# Build features

This part of the code handle the generation of features from the returns column. It also shrink the data type of the dataframe in order to save RAM.

In [ ]:
from src.feature_engineering import polars_features

df = polars_features(data, value_col='ret', date_col='date', target_col = 'ret')

print(f'Dataset RAM size before shrink : {df.estimated_size() / 1024**3:.2f}Gb')
df = shrink_polars(df)
print(f'Dataset RAM size after shrink : {df.estimated_size() / 1024**3:.2f}Gb')
 

df = df.to_pandas()


df.to_parquet(config.FEATURES_PATH_CLEAN, compression='zstd')

---
# Load Chen-Zimmerman Dataset

In [2]:
from src.data_loading import load_cz_monthly


# 1. Load and shrink

cz_daily = load_cz_monthly() # Include a one day shift

print(f'Dataset RAM size before shrink : {cz_daily.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
cz_daily = shrink(cz_daily)
print(f'Dataset RAM size after shrink : {cz_daily.memory_usage(index=True).sum() / 1024**3:.2f}Gb')


# 2. Index cz dataset on date

cz_indexed = cz_daily.set_index('date').sort_index()
cz_indexed.index = pd.to_datetime(cz_indexed.index).astype('datetime64[ms]') # Prepare to merge


# 3. Upload to .parquet 

cz_indexed.to_parquet(config.CZ_PATH_CLEAN, compression='zstd')
cz_indexed.head()

Initial Size of the dataset: (1140, 206)
Total Date range : 1926-01-30 00:00:00 -> 2020-12-31 00:00:00
Dataset shape after dropping column with less than 90.0% completion: (1140, 57)
Total column dropped so far : 149
Dataset shape after dropping highly correlated (corr_coef > 0.95) columns: (1140, 55)
Total column dropped so far : 151
Dataset RAM size before shrink : 0.01Gb
Dataset RAM size after shrink : 0.01Gb


,Beta,BetaFP,BetaTailRisk,BidAskSpread,CompEquIss,Coskewness,DivInit,DivOmit,DivSeason,DivYieldST,...,Size,Spinoff,std_turn,STreversal,VolMkt,VolSD,VolumeTrend,zerotrade,zerotradeAlt1,zerotradeAlt12
date,,,,,,,,,,,,,,,,,,,,,
1932-01-30,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-01-31,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-02-01,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-02-02,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-02-03,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856


---
# Import and Clean VIX data

In [ ]:
df_vix = fetch_VIX()
df_vix = pd.read_parquet(config.VIX_PATH_RAW)
df = clean_vix(df_vix)
df_vix = pd.read_parquet(config.VIX_PATH_CLEAN)
df_vix.head()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Data saved to /Users/dariofrey/Documents/Dario/Université/EPFL/MFE/MA-2/Machine Learning in Finance/ML-For-Finance-Project-LeoWunderli-329314-DarioFrey-344524/notebooks/../data/raw/vix.parquet
Saved as .parquet file to {config.VIX_PATH_CLEAN}


,vix,vix_ma_63,vix_ma_126,vix_ma_252
date,,,,
1990-12-31,0.2505,0.2611,0.2565,0.2305
1991-01-02,0.2638,0.2608,0.2573,0.2309
1991-01-03,0.2662,0.2607,0.2582,0.2312
1991-01-04,0.2793,0.2607,0.2590,0.2315
1991-01-07,0.2719,0.2607,0.2599,0.2318
